# Pandas for Data Engineering — A Practical Guide

This notebook demonstrates essential pandas operations for data engineering workflows, including data loading, type conversion, null handling, outlier filtering, date manipulation, feature creation, merging, and aggregation. It uses an e-commerce orders dataset (~369K rows) and a stores reference table to illustrate real-world data manipulation techniques.

### Preparing the Dataset

Before diving into analysis, we need to load the required libraries and import the raw data. This section covers importing pandas and reading the orders CSV file into a DataFrame.

In [0]:
# Import the pandas library — the primary data manipulation library in Python
import pandas as pd

In [0]:
# Load the orders dataset from the data directory
# The CSV file contains e-commerce order records with 29 columns
orders = pd.read_csv('../data/orders.csv')
orders.head()

### Data Structure Analysis

Once the data is loaded, we inspect its structure — column names, data types, null values, and duplicate rows — to understand what cleaning steps are needed.

In [0]:
# Display rows where the 'order_status' column has null values
# This helps identify missing data that may need to be handled
orders[
    orders['order_status'].isnull()
]

In [0]:
# List all column names in the DataFrame
orders.columns

In [0]:
# Drop the 'order_moment_delivered' column
# axis=1 indicates column-level operation (axis=0 would be row-level)
# inplace=True modifies the DataFrame in place without returning a copy
orders.drop('order_moment_delivered', axis=1, inplace=True)

In [0]:
# Count unique values in the 'order_status' column
# Equivalent to SQL: SELECT order_status, COUNT(*) FROM orders GROUP BY order_status
orders['order_status'].value_counts()

In [0]:
# The 'dtype' property shows the data type of a Series
# The 'astype()' method converts a column to a different data type
print(f"Before: {orders['order_id'].dtype}")
orders['order_id'] = orders['order_id'].astype(int)
print(f"After: {orders['order_id'].dtype}")

In [0]:
# Convert 'order_amount' from string to float for numeric operations
print(f"Before: {orders['order_amount'].dtype}")
orders['order_amount'] = orders['order_amount'].astype(float)
print(f"After: {orders['order_amount'].dtype}")

In [0]:
# Remove duplicate rows based on the 'order_id' column
# The 'subset' parameter specifies which columns to use for deduplication
orders.drop_duplicates(subset=['order_id'])

In [0]:
# Convert 'order_delivery_cost' from string to float
orders['order_delivery_cost'] = orders['order_delivery_cost'].astype(float)

In [0]:
# Fill null values in 'order_delivery_cost' with the column mean (rounded to 2 decimal places)
# The fillna() method replaces NaN values with the specified value
orders['order_delivery_cost'] = orders['order_delivery_cost'].fillna(round(orders['order_delivery_cost'].mean(), 2))

In [0]:
# Replace 'CANCELED' with 'CANCELADO' in the 'order_status' column
# The loc[] accessor allows label-based selection and assignment
orders.loc[
    (orders['order_status'] == 'CANCELED'), 'order_status'
] = 'CANCELADO'

### Anomalies & Outliers

Statistical summary and outlier detection help identify data quality issues. We use `describe()` to get distribution statistics and filter out extreme values that could skew analysis.

In [0]:
# Configure pandas to display float values with 2 decimal places
pd.set_option('display.float_format', '{:.2f}'.format)

In [0]:
# Generate descriptive statistics for the 'order_amount' column
# Shows count, mean, std, min, quartiles, and max
orders['order_amount'].describe()

In [0]:
# Filter out outlier orders with amount >= 1000
# These extreme values can distort aggregations and analysis
orders = orders[
    (orders['order_amount'] < 1000)    
]

In [0]:
# Re-check the descriptive statistics after removing outliers
orders['order_amount'].describe()

### Date Manipulation

Working with dates is essential for time-based analysis. We convert string timestamps to datetime objects and extract date components for grouping and filtering.

In [0]:
# Convert 'order_moment_created' to datetime, then extract just the date portion
# pd.to_datetime() parses string timestamps into datetime objects
# .dt.date extracts the date component (YYYY-MM-DD), discarding the time
orders['order_moment_created_date'] = pd.to_datetime(orders['order_moment_created']).dt.date

In [0]:
# Verify that the new date column was added to the DataFrame
orders.columns

In [0]:
# Display the newly created date column
orders['order_moment_created_date']

### Creating New Columns

Feature engineering — creating new columns from existing ones — is a core data preparation step. Here we combine multiple metric columns to derive a lead time metric.

In [0]:
# Display a summary of the DataFrame including column types and non-null counts
orders.info()

In [0]:
# Convert time metric columns from string to float for arithmetic operations
orders['order_metric_collected_time'] = orders['order_metric_collected_time'].astype(float)
orders['order_metric_production_time'] = orders['order_metric_production_time'].astype(float)

In [0]:
# Create a new column 'lead_time_product' by summing collection and production times
# This represents the total time from order collection to production completion
orders['lead_time_product'] = orders['order_metric_collected_time'] + orders['order_metric_production_time']
orders['lead_time_product'].describe()


In [0]:
# Filter orders where lead time exceeds 1000 (potential anomalies)
orders[
    (orders['lead_time_product']) > 1000
] 

### Grouping and Selection

Aggregating data by groups is fundamental for reporting and analysis. We use `groupby()` with `agg()` — similar to SQL's `GROUP BY` — to summarize order amounts by date and store.

In [0]:
# Group orders by date and store, then sum the order amounts
# Equivalent to SQL: SELECT order_moment_created_date, store_id, SUM(order_amount)
#                  FROM orders GROUP BY order_moment_created_date, store_id
orders.groupby(['order_moment_created_date', 'store_id']).agg({'order_amount':'sum'}).reset_index()

In [0]:
# Load the stores reference dataset
# encoding='latin-1' is used because the file contains non-UTF-8 characters (Portuguese accents)
stores = pd.read_csv('../data/stores.csv', encoding='latin-1')
stores.head()

In [0]:
# Extract store_id and store_name into a new DataFrame for joining
stores_name = stores[['store_id', 'store_name']].copy()

In [0]:
# Merge orders with store names using a left join on 'store_id'
# This enriches the orders data with human-readable store names
orders_store = orders.merge(stores_name, how='left', on='store_id')
orders_store.head()

In [0]:
# Group by date and store name, then sum order amounts
# This produces a daily revenue summary per store
order_amount_stores = orders_store.groupby(['order_moment_created_date', 'store_name']).agg({'order_amount':'sum'}).reset_index()

In [0]:
# Export the aggregated result to a CSV file in the data directory
# index=False prevents pandas from writing the row index as a column
order_amount_stores.to_csv('../data/order_amount_stores.csv', index=False)

In [0]:
# Display the final aggregated DataFrame
order_amount_stores.head()